# Homework 3  

##### Compute all 6 Keplerian elements for both Vectors 1 and 2

In [87]:
from standards import *

v1_pos = Vector3(326151.080726, 6077471.251787, 2944583.918767)
v1_vel = Vector3(-7455.178720, -482.482572, 1910.883434)
v2_pos = Vector3(327000.0, 6077600.0, 2944280.0)
v2_vel = Vector3(-7454.0, -483.0, 1911.0)

v1_kep = KeplerianElements(v1_pos, v1_vel)
v2_kep = KeplerianElements(v2_pos, v2_vel)

##### Compute the **ECI → UVW** transformation matrix $[\mathrm{T}_{\text{ECI}}^{\text{UVW}}]$ using **Vector 1**
##### Note: this transformation is done through method 2: Define basis vectors of new frame as components of unit vectors expressed in original reference frame
##### UVW Basis Vectors from ECI position and velocity (Found on slide 32 but these should be intuitive)
$\hat{U} = \frac{\vec{r}}{|\vec{r}|}$

$\hat{W} = \frac{\vec{r} \times \vec{v}}{|\vec{r} \times \vec{v}|}$

$\hat{V} = \hat{W} \times \hat{U}$  

<img src="ECI_UVW.png" alt="ECI to UVW" width="500"/>

And to get rotated coordinates, just multiply the rotational matrix by the vector:  
<img src="Get_Rotated_Coordinates.png" alt="ECI to UVW" width="500"/>

In [88]:
import numpy as np

def compute_matrix_eci_to_uvw(pos_, vel_: Vector3):
    U_hat = pos_/abs(pos_.magnitude())
    W_hat = (pos_.cross(vel_))/abs((pos_.cross(vel_)).magnitude())
    V_hat = W_hat.cross(U_hat)
    # rotation matrix:
    R = np.array([
        [U_hat.x, U_hat.y, U_hat.z],
        [V_hat.x, V_hat.y, V_hat.z],
        [W_hat.x, W_hat.y, W_hat.z]
    ])
    # note - to translate backwards from uvw to eci use the inverse of this matrix
    return R


In [89]:
eci_to_uvw_matrix = compute_matrix_eci_to_uvw(v1_pos, v1_vel)
print(eci_to_uvw_matrix)

[[ 0.04823928  0.89888664  0.43551784]
 [-0.96704342 -0.06710302  0.24560988]
 [ 0.25       -0.4330127   0.8660254 ]]


### Show that $[\mathrm{T}_\mathrm{ECI}^\mathrm{UVW}] \vec{r}_1 - [\mathrm{T}_\mathrm{ECI}^\mathrm{UVW}] \vec{r}_2 = [\mathrm{T}_\mathrm{ECI}^\mathrm{UVW}] (\vec{r}_1 - \vec{r}_2)$  
##### Mathematically:  
<img src="Derivation_1.png" alt="Derivation_1"/>  

##### Numerically:  

$[\mathrm{T}_\mathrm{ECI}^\mathrm{UVW}] \vec{r}_1 - [\mathrm{T}_\mathrm{ECI}^\mathrm{UVW}] \vec{r}_2$:

In [90]:
r1_pos_np = v1_pos.get_np_vector()
r2_pos_np = v2_pos.get_np_vector()

# Note - matrix multiplication in numpy uses the @ sign
print(eci_to_uvw_matrix @ r1_pos_np - eci_to_uvw_matrix @ r2_pos_np)

[[-24.31926011]
 [904.22664159]
 [106.72116603]]


$[\mathrm{T}_\mathrm{ECI}^\mathrm{UVW}] (\vec{r}_1 - \vec{r}_2)$:

In [91]:
# Note - matrix multiplication in numpy uses the @ sign
print(eci_to_uvw_matrix @ (r1_pos_np - r2_pos_np))

[[-24.31926011]
 [904.22664159]
 [106.72116603]]


### Interpret the results of the previous problem. What does $[\mathrm{T}_\mathrm{ECI}^\mathrm{UVW}] (\vec{r}_1 - \vec{r}_2)$ mean?

- The transformation matrix is transforming ECI vectors to the Vector 1 UVW (or RIC) frame. 
- We are taking the difference between the two vectors, then transforming the difference into the Vector 1 frame. 
- The other way to think about it is transforming both Vector 1 and Vector 2 into the Vector 1 frame. 
- If we subtract the magnitude of Vector 1 position from the radial component of $[\mathrm{T}_\mathrm{ECI}^\mathrm{UVW}] (\vec{r}_1)$, we would end up with the zero vector which means that our origin would be the position of Vector 1. 
- Essentially, the result of $[\mathrm{T}_\mathrm{ECI}^\mathrm{UVW}] (\vec{r}_1 - \vec{r}_2)$ can give us the distance between the two satellites in the UVW (or RIC) frame and show us the distance between the two, in the intrack, radial, and crosstrack directions. 
- Based off the calculations below, the absolute distance between the two objects is 910.8 m. 
- If we had an array of Vectors for each object over time, we could plot the relative motion between the two objects over time

In [92]:
v1_uvw_np = eci_to_uvw_matrix @ r1_pos_np
v1_uvw = Vector3(float(v1_uvw_np[0][0]), float(v1_uvw_np[1][0]), float(v1_uvw_np[2][0]))
v2_uvw_np = eci_to_uvw_matrix @ r2_pos_np
v2_uvw = Vector3(float(v2_uvw_np[0][0]), float(v2_uvw_np[1][0]), float(v2_uvw_np[2][0]))
pos_diff = v2_uvw - v1_uvw
print("Relative to Vector 1, Vector 2 is:")
print("Radial distance:      " + str(pos_diff.x) + " m")
print("Intrack distance:     " + str(pos_diff.y) + " m")
print("Crosstrack distance:  " + str(pos_diff.z) + " m")
print("Absolute distance:    " + str(pos_diff.magnitude()) + " m")

Relative to Vector 1, Vector 2 is:
Radial distance:      24.31926010735333 m
Intrack distance:     -904.2266415915219 m
Crosstrack distance:  -106.72116602631286 m
Absolute distance:    910.8274551494743 m


### Suppose you have a sensor on a satellite, mounted in the V-W plane of the UVW system of problem 3, and rotated +30° about the U axis. Call this the “instrument frame.” Derive an expression for the transformation from ECI→Instrument frame  
<img src="Derivation_2.png" alt="Derivation_2"/>  


## Given the Keplerian elements:

- **Semi-major axis (a)**: 78,000,000.0 meters  
- **Eccentricity (e)**: 0.001  
- **Inclination (i)**: 98.6°  
- **Right ascension of ascending node (Ω)**: 30.0°  
- **Argument of perigee (ω_p)**: 40.0°  
- **True anomaly (ν)**: 50.087853°  

## Compute the corresponding ECI position and velocity vectors  

##### First compute perifocal vectors so we can rotate them to ECI later:

In [93]:
kep = KeplerianElements(None, None)
kep.a = 7800000.0
kep.ecc = 0.001
kep.inc = math.radians(98.6)
kep.raan = math.radians(30)
kep.argp = math.radians(40)
kep.ta = math.radians(50.087853)
pos_perifocal, vel_perifocal = compute_perifocal_coordinates(kep)
print("Perifocal Vectors:")
print("pos: " + str(pos_perifocal))
print("vel: " + str(vel_perifocal))

# Verify this:

print("Verify this math...")
p = kep.a*(1-math.pow(kep.ecc,2))
r = p/(1+kep.ecc*math.cos(kep.ta))
rp = (r*math.cos(kep.ta), r*math.sin(kep.ta), 0)
vp = (-math.sqrt(kep.mu/p)*math.sin(kep.ta), math.sqrt(kep.mu/p)*(kep.ecc+math.cos(kep.ta)), 0)
print("Calculated position: " + str(rp))
print("Calculated velocity: " + str(vp))
print("Expected position: (5001361.721974,5978985.122490, 0.000000)")
print("Expected velocity: (-5483.194701, 4593.786567, 0.000000)")


Perifocal Vectors:
pos: Vector3(x=5001361.689315008, y=5978985.149839366, z=0)
vel: Vector3(x=-5483.194725650961, y=4593.786537388184, z=0)
Verify this math...
Calculated position: (5001361.689315008, 5978985.149839366, 0)
Calculated velocity: (-5483.194725650961, 4593.786537388184, 0)
Expected position: (5001361.721974,5978985.122490, 0.000000)
Expected velocity: (-5483.194701, 4593.786567, 0.000000)


##### Mathematically derive the rotation matrix from perifocal to ECI  
##### Note: This is done using method 1: Rotation about a basis axis
<img src="Derivation_3.png" alt="Derivation_3"/>  

##### Now code it up and calculate the rotation matrix and use it to rotate the perifocal position and velocity vectors:

In [94]:
# compute rotation perifocal to eci
def compute_rotaton_perifocal_to_eci(raan: float, i: float, argp: float, perifocal_pos: Vector3, perifocal_vel: Vector3) -> tuple[np.array, Vector3, Vector3]:
    # rotation matrix:
    R = np.array([
        [math.cos(raan)*math.cos(argp)-math.sin(raan)*math.sin(argp)*math.cos(i), -math.cos(raan)*math.sin(argp)-math.sin(raan)*math.cos(argp)*math.cos(i), math.sin(raan)*math.sin(i)],
        [math.sin(raan)*math.cos(argp)+math.cos(raan)*math.sin(argp)*math.cos(i), -math.sin(raan)*math.sin(argp)+math.cos(raan)*math.cos(argp)*math.cos(i), -math.cos(raan)*math.sin(i)],
        [math.sin(i)*math.sin(argp), math.sin(i)*math.cos(argp), math.cos(i)]
    ])
    pos_np = ([
        [perifocal_pos.x],
        [perifocal_pos.y],
        [perifocal_pos.z]
    ])
    vel_np = ([
        [perifocal_vel.x],
        [perifocal_vel.y],
        [perifocal_vel.z]
    ])
    eci_pos_np = R @ pos_np
    eci_vel_np = R @ vel_np
    return R, Vector3(float(eci_pos_np[0][0]), float(eci_pos_np[1][0]), float(eci_pos_np[2][0])) ,Vector3(float(eci_vel_np[0][0]), float(eci_vel_np[1][0]), float(eci_vel_np[2][0]))

In [95]:
R, eci_pos, eci_vel = compute_rotaton_perifocal_to_eci(kep.raan, kep.inc, kep.argp, pos_perifocal, vel_perifocal)
print("Rotation Matrix:")
print(R)
print("ECI Vectors:")
print("pos: " + str(eci_pos))
print("vel: " + str(eci_vel))

Rotation Matrix:
[[ 0.71147368 -0.49939504  0.49437819]
 [ 0.29978032 -0.42059764 -0.85628814]
 [ 0.63556035  0.75743133 -0.14953534]]
ECI Vectors:
pos: Vector3(x=572461.6851520538, y=-1015437.2094471643, z=7707337.871278939)
vel: Vector3(x=-6195.262946533725, y=-3575.8896461590466, z=-5.423310266261069)
